# Delivery Time Estimation

Regression coursework examining delivery duration through timestamp processing, feature engineering, exploratory analysis, model building, and residual diagnostics.

## Portfolio edition

This notebook was migrated from Danish Ali Syed's Colab learning work. It includes educational starter material and exercises; it is not presented as a production deployment or as wholly original course content.

Saved outputs and execution state have been cleared. Personal Drive links, mounts, and cloud bucket references have been removed. Supply the authorised input data locally; see README.md. The notebook was reviewed statically, not rerun during migration. Results described in the source are historical and have not been independently reproduced.


# Order Delivery Time Prediction

## Objectives
The objective of this assignment is to build a regression model that predicts the delivery time for orders placed through Porter. The model will use various features such as the items ordered, the restaurant location, the order protocol, and the availability of delivery partners.

The key goals are:
- Predict the delivery time for an order based on multiple input features
- Improve delivery time predictions to optimiae operational efficiency
- Understand the key factors influencing delivery time to enhance the model's accuracy

## Data Pipeline
The data pipeline for this assignment will involve the following steps:
1. **Data Loading**
2. **Data Preprocessing and Feature Engineering**
3. **Exploratory Data Analysis**
4. **Model Building**
5. **Model Inference**

## Data Understanding
The dataset contains information on orders placed through Porter, with the following columns:

| Field                     | Description                                                                                 |
|---------------------------|---------------------------------------------------------------------------------------------|
| market_id                 | Integer ID representing the market where the restaurant is located.                         |
| created_at                | Timestamp when the order was placed.                                                        |
| actual_delivery_time      | Timestamp when the order was delivered.                                                     |
| store_primary_category    | Category of the restaurant (e.g., fast food, dine-in).                                      |
| order_protocol            | Integer representing how the order was placed (e.g., via Porter, call to restaurant, etc.). |
| total_items               | Total number of items in the order.                                                         |
| subtotal                  | Final price of the order.                                                                   |
| num_distinct_items        | Number of distinct items in the order.                                                      |
| min_item_price            | Price of the cheapest item in the order.                                                    |
| max_item_price            | Price of the most expensive item in the order.                                              |
| total_onshift_dashers     | Number of delivery partners on duty when the order was placed.                              |
| total_busy_dashers        | Number of delivery partners already occupied with other orders.                             |
| total_outstanding_orders  | Number of orders pending fulfillment at the time of the order.                              |
| distance                  | Total distance from the restaurant to the customer.                                         |


## **Importing Necessary Libraries**

In [ ]:
# Import essential libraries for data manipulation and analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler

## **1. Loading the data**
Load 'porter_data_1.csv' as a DataFrame

In [ ]:
from pathlib import Path
# Importing the file porter_data_1.csv
BASE_PATH = str(Path("data").resolve())
porter = pd.read_csv(f"{BASE_PATH}/porter_data_1.csv")
porter.head()

In [ ]:
porter.shape

In [ ]:
porter.info()

## **2. Data Preprocessing and Feature Engineering** <font color = red>[15 marks]</font> <br>

#### **2.1 Fixing the Datatypes**  <font color = red>[5 marks]</font> <br>
The current timestamps are in object format and need conversion to datetime format for easier handling and intended functionality

##### **2.1.1** <font color = red>[2 marks]</font> <br>
Convert date and time fields to appropriate data type

In [ ]:
# Convert 'created_at' and 'actual_delivery_time' columns to datetime format
porter['created_at'] = pd.to_datetime(porter['created_at'])
porter['actual_delivery_time'] = pd.to_datetime(porter['actual_delivery_time'])

porter.dtypes

##### **2.1.2**  <font color = red>[3 marks]</font> <br>
Convert categorical fields to appropriate data type

In [ ]:
porter.nunique()

In [ ]:
# Convert categorical features to category type
porter.select_dtypes(include='object').columns

In [ ]:
cat_cols = ['market_id', 'store_primary_category','order_protocol']

for col in cat_cols:
    porter[col] = porter[col].astype('category')
porter.dtypes

#### **2.2 Feature Engineering** <font color = red>[5 marks]</font> <br>
Calculate the time taken to execute the delivery as well as extract the hour and day at which the order was placed

##### **2.2.1** <font color = red>[2 marks]</font> <br>
Calculate the time taken using the features `actual_delivery_time` and `created_at`

In [ ]:
# Calculate time taken in minutes
porter['delivery_time'] = (
    porter['actual_delivery_time'] - porter['created_at']
).dt.total_seconds() / 60

In [ ]:
porter[['created_at', 'actual_delivery_time', 'delivery_time']].head()

In [ ]:
porter['delivery_time'].describe()

In [ ]:
porter.info()

##### **2.2.2** <font color = red>[3 marks]</font> <br>
Extract the hour at which the order was placed and which day of the week it was. Drop the unnecessary columns.

In [ ]:
# Extract the hour and day of week from the 'created_at' timestamp
# Extract hour of order
porter['order_hour'] = porter['created_at'].dt.hour
# Extract day of week (0 = Monday, 6 = Sunday)
porter['order_day'] = porter['created_at'].dt.dayofweek

# Create a categorical feature 'isWeekend'
# Create weekend feature
porter['is_weekend'] = porter['order_day'].apply(lambda x: 1 if x >= 5 else 0)
# Convert to categorical
porter['is_weekend'] = porter['is_weekend'].astype('category')

In [ ]:
porter.dtypes

In [ ]:
# Drop unnecessary columns
porter = porter.drop(['created_at', 'actual_delivery_time', 'order_day'], axis=1)

In [ ]:
porter.info()

In [ ]:
porter.head()

#### **2.3 Creating training and validation sets** <font color = red>[5 marks]</font> <br>

##### **2.3.1** <font color = red>[2 marks]</font> <br>
 Define target and input features

In [ ]:
# Define target variable (y) and features (X)
# Target variable
y = porter['delivery_time']
# Features
X = porter.drop('delivery_time', axis=1)

##### **2.3.2** <font color = red>[3 marks]</font> <br>
 Split the data into training and test sets

In [ ]:
# Split data into training and testing sets
import sklearn
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(porter, train_size=0.7, random_state=100)
print(df_train.shape)
print(df_test.shape)

## **3. Exploratory Data Analysis on Training Data** <font color = red>[20 marks]</font> <br>
1. Analyzing the correlation between variables to identify patterns and relationships
2. Identifying and addressing outliers to ensure the integrity of the analysis
3. Exploring the relationships between variables and examining the distribution of the data for better insights

#### **3.1 Feature Distributions** <font color = red> [7 marks]</font> <br>


In [ ]:
# Define numerical and categorical columns for easy EDA and data manipulation

print(df_train.info())
print('---')

num_cols = df_train.select_dtypes(include=['number']).columns
cat_cols = df_train.select_dtypes(include=['category']).columns
print("Numerical columns:", list(num_cols))
print("Categorical columns:", list(cat_cols))

##### **3.1.1** <font color = red>[3 marks]</font> <br>
Plot distributions for numerical columns in the training set to understand their spread and any skewness

In [ ]:
# Plot distributions for all numerical columns (Pairplot for training data)
sns.pairplot(df_train, y_vars=['delivery_time'], height=3)
plt.suptitle('Pairplot of Numerical Features vs. Delivery Time (Training Data)', y=1.02)
plt.show()

In [ ]:
# Plot distributions for all numerical columns
df_train[num_cols].hist(figsize=(18, 12), bins=30)
plt.tight_layout()
plt.show()

It was observed that most variables such as subtotal, min_item_price, max_item_price, total_items, and num_distinct_items exhibit right-skewed distributions, indicating that most orders have lower values with a few high-value outliers.

Variables like total_onshift_dashers, total_busy_dashers, and total_outstanding_orders also show right skewness, reflecting operational variability with occasional peak loads.

The distance variable appears to follow a relatively normal distribution, suggesting most deliveries occur within a common range of distances. Overall, the dataset contains skewed numerical features and potential outliers.

##### **3.1.2** <font color = red>[2 marks]</font> <br>
Check the distribution of categorical features

In [ ]:
# Distribution of categorical columns
plt.figure(figsize=(20,8))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(4, 6, i)
    df_train[col].value_counts().sort_index().plot(kind='bar')
    plt.title(col)
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##### **3.1.3** <font color = red>[2 mark]</font> <br>
Visualise the distribution of the target variable to understand its spread and any skewness

In [ ]:
# Distribution of time_taken
plt.figure(figsize=(10, 6))
sns.histplot(df_train['delivery_time'], bins=30, kde=True)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

#### **3.2 Relationships Between Features** <font color = red>[3 marks]</font> <br>

##### **3.2.1** <font color = red>[3 marks]</font> <br>
Scatter plots for important numerical and categorical features to observe how they relate to `time_taken`

In [ ]:
# Scatter plot to visualise the relationship between delivery_time and numerical features
plt.figure(figsize=(15,15))
num_cols_filtered = [col for col in num_cols if col != 'delivery_time']

for i, col in enumerate(num_cols_filtered, 1):
    plt.subplot(4, 3, i)
    plt.scatter(df_train[col], df_train['delivery_time'])
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel('delivery_time')

plt.tight_layout()
plt.show()

In [ ]:
# Show the distribution of time_taken for different hours
plt.figure(figsize=(10, 6))
sns.boxplot(x=df_train['order_hour'], y=df_train['delivery_time'])
plt.title('Delivery Time by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Delivery Time (minutes)')
plt.show()

#### **3.3 Correlation Analysis** <font color = red>[5 marks]</font> <br>
Check correlations between numerical features to identify which variables are strongly related to `time_taken`

##### **3.3.1** <font color = red>[3 marks]</font> <br>
Plot a heatmap to display correlations

In [ ]:
# Plot the heatmap of the correlation matrix
plt.figure(figsize=(15, 10))
df_numerical = df_train.select_dtypes(include=[np.number])
sns.heatmap(df_numerical.corr(), annot = True, cmap = "YlGnBu")
#sns.heatmap(df_train.corr(), annot = True, cmap = "YlGnBu")
plt.show()

In [ ]:
# Calculate correlations of numerical features with 'delivery_time'
# Get numeric columns
current_num_cols = df_train.select_dtypes(include=['number']).columns.tolist()
# Remove target column
features_for_corr = [col for col in current_num_cols if col != 'delivery_time']
# Calculate correlation with target
correlation_with_delivery_time = df_train[features_for_corr].corrwith(df_train['delivery_time'])
# Convert to DataFrame since the pandas Series correlation_with_delivery_time returns values without column name
corr_df = correlation_with_delivery_time.to_frame(name='correlation')
# Add absolute correlation column
corr_df['abs_correlation'] = corr_df['correlation'].abs()
# Sort by absolute correlation (descending = strongest first)
corr_df = corr_df.sort_values(by='abs_correlation', ascending=False)
# Display result
display(corr_df)

##### **3.3.2** <font color = red>[2 marks]</font> <br>
Drop the columns with weak correlations with the target variable

In [ ]:
# Drop 3-5 weakly correlated columns from training dataset

columns_to_drop = ['min_item_price', 'total_onshift_dashers', 'total_busy_dashers']

df_train = df_train.drop(columns=columns_to_drop, errors='ignore')

print(f"Columns successfully dropped from df_train: {columns_to_drop}")
print(f"Remaining columns in df_train: {df_train.columns.tolist()}")

In [ ]:
df_train.head()

#### **3.4 Handling the Outliers** <font color = red>[5 marks]</font> <br>



##### **3.4.1** <font color = red>[2 marks]</font> <br>
Visualise potential outliers for the target variable and other numerical features using boxplots

In [ ]:
# Boxplot for time_taken
plt.figure(figsize=(10, 6))
sns.boxplot(y=df_train['delivery_time'])
plt.title('Boxplot of Delivery Time (Training Data)')
plt.ylabel('Delivery Time (minutes)')
plt.show()

##### **3.4.2** <font color = red>[3 marks]</font> <br>
Handle outliers present in all columns

In [ ]:
# Handle outliers
# Function to cap outliers using the IQR method
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df

# Function to remove outliers using the IQR method
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_filtered

# Get numerical columns X from df_train excluding 'delivery_time'
num_cols_for_capping = df_train.select_dtypes(include=[np.number]).columns.drop('delivery_time').tolist()

# Apply outlier capping to numerical features
print(f"Shape of df_train before outlier capping: {df_train.shape}")
for col in num_cols_for_capping:
    df_train = cap_outliers_iqr(df_train, col)
print(f"Shape of df_train after outlier capping: {df_train.shape}")

# Apply outlier removal to 'delivery_time' (target variable)
print(f"Shape of df_train before delivery_time outlier removal: {df_train.shape}")
df_train = remove_outliers_iqr(df_train, 'delivery_time')
print(f"Shape of df_train after delivery_time outlier removal: {df_train.shape}")

In [ ]:
# Boxplot for time_taken
plt.figure(figsize=(10, 6))
sns.boxplot(y=df_train['delivery_time'])
plt.title('Boxplot of Delivery Time (Training Data)')
plt.ylabel('Delivery Time (minutes)')
plt.show()

## **4. Exploratory Data Analysis on Validation Data** <font color = red>[optional]</font> <br>
Optionally, perform EDA on test data to see if the distribution match with the training data

In [ ]:
# Define numerical and categorical columns for easy EDA and data manipulation in the test set
# Numerical columns
num_cols_test = df_test.select_dtypes(include=[np.number]).columns.tolist()

# Categorical columns (if any remain)
cat_cols_test = df_test.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical columns:", num_cols_test)
print("Categorical columns:", cat_cols_test)

#### **4.1 Feature Distributions**


##### **4.1.1**
Plot distributions for numerical columns in the validation set to understand their spread and any skewness

In [ ]:
# Plot distributions for all numerical columns in the test set
df_test[num_cols_test].hist(figsize=(18, 12), bins=30)
plt.tight_layout()
plt.show()

##### **4.1.2**
Check the distribution of categorical features

In [ ]:
# Distribution of categorical columns in the test set
plt.figure(figsize=(20,8))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(4, 6, i)
    df_test[col].value_counts().sort_index().plot(kind='bar')
    plt.title(col)
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

##### **4.1.3**
Visualise the distribution of the target variable to understand its spread and any skewness

In [ ]:
# Distribution of time_taken in the test set
plt.figure(figsize=(10, 6))
sns.histplot(df_test['delivery_time'], bins=30, kde=True)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

#### **4.2 Relationships Between Features**
Scatter plots for numerical features to observe how they relate to each other, especially to `time_taken`

In [ ]:
# Scatter plot to visualise the relationship between time_taken and other features in the test set
plt.figure(figsize=(15,15))
num_cols_filtered_test = [col for col in num_cols_test if col != 'delivery_time']

for i, col in enumerate(num_cols_filtered_test, 1):
    plt.subplot(4, 3, i)
    plt.scatter(df_test[col], df_test['delivery_time'])
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel('delivery_time')

plt.tight_layout()
plt.show()

#### **4.3** Drop the columns with weak correlations with the target variable

In [ ]:
# Drop the weakly correlated columns from training dataset
# Columns identified from training data as weakly correlated
columns_to_drop_test = ['min_item_price', 'total_onshift_dashers', 'total_busy_dashers']


# Drop the same columns from test data
df_test = df_test.drop(columns=columns_to_drop_test, errors='ignore')

print(f"Columns dropped from df_test: {columns_to_drop_test}")
print(f"Updated df_test shape: {df_test.shape}")
print(f"Updated df_train shape: {df_train.shape}")
display(df_test.head())

## **5. Model Building** <font color = red>[15 marks]</font> <br>

#### **Import Necessary Libraries**

In [ ]:
# Import libraries
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import RFE

In [ ]:
# X_train, y_train
y_train = df_train['delivery_time']
X_train = df_train.drop(columns=['delivery_time'])
y_train.head()

In [ ]:
df_train.shape

In [ ]:
y_train.shape

In [ ]:
X_train.shape

In [ ]:
print(df_test.info())
print('---')

num_cols_test = df_test.select_dtypes(include=['number']).columns
cat_cols_test = df_test.select_dtypes(include=['category']).columns
print("Numerical columns:", list(num_cols_test))
print("Categorical columns:", list(cat_cols_test))

#### **5.1 Feature Scaling** <font color = red>[3 marks]</font> <br>

In [ ]:
# Apply scaling to the numerical columns
scaler = MinMaxScaler()
# Get all numerical columns currently in X_train
all_numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
X_train[all_numeric_cols] = scaler.fit_transform(X_train[all_numeric_cols])
X_train.head()

Note that linear regression is agnostic to feature scaling. However, with feature scaling, we get the coefficients to be somewhat on the same scale so that it becomes easier to compare them.

In [ ]:
# Handle high-cardinality column
cat_cols = ['market_id', 'store_primary_category', 'order_protocol']

def preprocess_data(df, top_categories, fit_columns=None):
    df = df.copy()

    # handle high-cardinality column
    df['store_primary_category'] = df['store_primary_category'].apply(
        lambda x: x if x in top_categories else 'Other'
    )

    # one-hot encode
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    # keep is_weekend numeric
    if 'is_weekend' in df.columns:
        df['is_weekend'] = df['is_weekend'].astype(int)

    # align columns if train columns are provided
    if fit_columns is not None:
        df = df.reindex(columns=fit_columns, fill_value=0)

    return df

In [ ]:
top_categories = X_train['store_primary_category'].value_counts().head(10).index
X_train = preprocess_data(X_train, top_categories)
X_train.info()

#### **5.2 Build a linear regression model** <font color = red>[5 marks]</font> <br>

You can choose from the libraries *statsmodels* and *scikit-learn* to build the model.

In [ ]:
# Create/Initialise the model

#adding all variables to the model
#looking at all the columns in the data
porter.columns

In [ ]:
X_train.columns

In [ ]:
X_train_copy1 = X_train.copy()
y_train_copy1 = y_train.copy()

X_train_copy2 = X_train.copy()
y_train_copy2 = y_train.copy()

In [ ]:
#build a model with all variables
X_train_sm = sm.add_constant(X_train_copy1)
#create model
lr = sm.OLS(y_train_copy1, X_train_sm)
#fit
lr_model = lr.fit()
#params
print(lr_model.summary())

#A low p-value indicates that the variable is statistically significant in explaining the dependent variable. All pvalues are below 0.05 so significant

We could drop some variables analyzing Significance (p_values) and VIF


A low p-value indicates that the variable is statistically significant in explaining the dependent variable. All pvalues are below 0.05 so significant.

Now

VIF
The higher the variable is related to other variables the higher the VIF
We usually want VIF of less than 5

VIF of 5 means 5 =1/(1-r2) 1-r2 =0.2 r2 =0.8 which means almost 80 of variations of this variable is explained by all other variable variations

In [ ]:
#Checking VIF
#create a dataframe that will contain the names of all the feature variables and their respected VIF
VIF = pd.DataFrame()
VIF['Features'] = X_train_copy1.columns
VIF['VIF'] = [variance_inflation_factor(X_train_copy1.values, i) for i in range(X_train_copy1.shape[1])]
VIF['VIF'] = round(VIF['VIF'], 2)
VIF = VIF.sort_values(by = "VIF", ascending = False)
VIF

In [ ]:
#KEPT subtotal correlation → 0.41 ✅ strongest & distance correlation → 0.45 ✅ strongest
columns_to_drop = ['total_items', 'num_distinct_items', 'max_item_price']
X_AdjVIF = X_train_copy1.drop(columns_to_drop, axis=1)

In [ ]:
#create another model
#build a model with all variables
X_train_sm2 = sm.add_constant(X_AdjVIF)
#create model
lr2 = sm.OLS(y_train_copy1, X_train_sm2)
#fit
lr_model2 = lr2.fit()
#params
print(lr_model2.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF
VIF2 = pd.DataFrame()
VIF2['Features'] = X_AdjVIF.columns
VIF2['VIF'] = [variance_inflation_factor(X_AdjVIF.values, i) for i in range(X_AdjVIF.shape[1])]
VIF2['VIF'] = round(VIF2['VIF'], 2)
VIF2 = VIF2.sort_values(by = "VIF", ascending = False)
VIF2

Traded slightly lower R² for much more stable & reliable model

In [ ]:
#Residue = y_train - predicated values of y
y_train_pred = lr_model2.predict(X_train_sm2)
y_train_pred

In [ ]:
res = y_train_copy1 - y_train_pred
sns.distplot(res)

It is centered around 0 and normal distributed

In [ ]:
# Train the model using the training data
# Train the model
lmT = LinearRegression()
lmT.fit(X_AdjVIF, y_train_copy1)

print("Intercept:", lmT.intercept_)
print("Coefficients:", lmT.coef_)

coef_dfT = pd.DataFrame({
    'Feature': X_AdjVIF.columns,
    'Coefficient': lmT.coef_
})

display(coef_dfT)

In [ ]:
X_train_copy1.columns

In [ ]:
X_train.columns

In [ ]:
df_train.columns

In [ ]:
#Scaling test
#1. create a list of only numeric vars
num_vars_test = df_train.select_dtypes(include=[np.number]).columns.tolist()
num_vars_test = [col for col in num_vars_test if col != 'delivery_time']
print(num_vars_test)

In [ ]:
#2. Fit on data
df_test[num_vars_test] = scaler.transform(df_test[num_vars_test])
df_test.head()

In [ ]:
# X_test, y_test
y_test = df_test['delivery_time']
X_test = df_test.drop(columns=['delivery_time'])
y_test.head()

In [ ]:
X_test.head()

In [ ]:
X_test = preprocess_data(X_test, top_categories, fit_columns=X_train.columns)
X_test.info()

In [ ]:
X_test_copy1 = X_test.copy()
y_test_copy1 = y_test.copy()

X_test_copy2 = X_test.copy()
y_test_copy2 = y_test.copy()

In [ ]:
# Make predictions
X_test_sm = sm.add_constant(X_test_copy1[X_AdjVIF.columns], has_constant='add')

y_test_pred = lr_model2.predict(X_test_sm)
y_test_pred

In [ ]:
# Find results for evaluation metrics
# MSE
mse = mean_squared_error(y_test_copy1, y_test_pred)

# RMSE (calculated from MSE, as 'squared')
rmse = np.sqrt(mse)

# MAE
mae = mean_absolute_error(y_test_copy1, y_test_pred)

# R²
r2 = r2_score(y_test_copy1, y_test_pred)

print("RMSE:", rmse)
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

✅ RMSE = 5.75

- On average, our predictions are off by ~5.75 minutes

✔ Very reasonable for delivery time

✅ MAE = 4.41

- Average absolute error = ~4.4 minutes

✔ Even better — most predictions are close

✅ R² = 0.619

- Model explains:

~61.9% of variation in delivery time

✔ This is GOOD for real-world data

In [ ]:
# Plotting y_test and y_pred to understand the spread.
fig = plt.figure()
plt.scatter(y_test_copy1,y_test_pred)
fig.suptitle('y_test vs y_pred', fontsize=20)              # Plot heading
plt.xlabel('y_test', fontsize=18)                          # X-label
plt.ylabel('y_pred', fontsize=16)                          # Y-label

Note that we have 12 (depending on how you select features) training features. However, not all of them would be useful. Let's say we want to take the most relevant 8 features.

We will use Recursive Feature Elimination (RFE) here.

For this, you can look at the coefficients / p-values of features from the model summary and perform feature elimination, or you can use the RFE module provided with *scikit-learn*.

In [ ]:
X_train_copy2.info()

#### **5.3 Build the model and fit RFE to select the most important features** <font color = red>[7 marks]</font> <br>

For RFE, we will start with all features and use
the RFE method to recursively reduce the number of features one-by-one.

After analysing the results of these iterations, we select the one that has a good balance between performance and number of features.

In [ ]:
X_trainRFE = X_train_copy2
y_trainRFE = y_train_copy2

In [ ]:
# Loop through the number of features and test the model
#Running RFE with the output number of the variable equal to 15
lmrfe = LinearRegression()
lmrfe.fit(X_trainRFE, y_trainRFE)


rfe = RFE(lmrfe, n_features_to_select=15)
#running RFE ... n = no. of variables you need e.g 15)
rfe = rfe.fit(X_trainRFE, y_trainRFE)

In [ ]:
#RFE has support feature and ranking feature >> Tells if the feature was selected or not and Ranking is for all the features and is ranked in terms of importances >> Rank = 1 (best features)
list(zip(X_trainRFE.columns, rfe.support_, rfe.ranking_))

In [ ]:
selected_features  = X_trainRFE.columns[rfe.support_] # support is true
print(selected_features)

In [ ]:
X_trainRFE.columns[~rfe.support_] # support is not true

In [ ]:
rfe_results = pd.DataFrame({
    'Feature': X_trainRFE.columns,
    'Selected': rfe.support_,
    'Rank': rfe.ranking_
})

print(rfe_results.sort_values(by=['Selected', 'Rank'], ascending=[False, True]))

In [ ]:
# Build the final model with selected number of features
X_trainRFET = X_trainRFE[selected_features]
X_trainRFET.head()

In [ ]:
#Adding constant
X_trainRFET_sm = sm.add_constant(X_trainRFET)
lmRFET = sm.OLS(y_trainRFE, X_trainRFET_sm).fit()
print(lmRFET.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF
VIFRFET = pd.DataFrame()
XRFET = X_trainRFET_sm.drop(['const'], axis=1)
VIFRFET['Features'] = XRFET.columns
VIFRFET['VIF'] = [variance_inflation_factor(XRFET.values, i) for i in range(XRFET.shape[1])]
VIFRFET['VIF'] = round(VIFRFET['VIF'], 2)
VIFRFET = VIFRFET.sort_values(by = "VIF", ascending = False)
VIFRFET

In [ ]:
#KEPT subtotal correlation → 0.41 ✅ strongest & distance correlation → 0.45 ✅ strongest
columns_to_drop_rfet = ['total_items', 'num_distinct_items']
X_trainRFE_new = X_trainRFET_sm.drop(columns_to_drop_rfet, axis = 1)

In [ ]:
#create another model
X_trainRFE_new_sm = sm.add_constant(X_trainRFE_new)
lmRFET_new = sm.OLS(y_trainRFE, X_trainRFE_new_sm).fit()
print(lmRFET_new.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF
VIFRFET2 = pd.DataFrame()
XRFET2 = X_trainRFE_new_sm.drop(['const'], axis=1)
VIFRFET2['Features'] = XRFET2.columns
VIFRFET2['VIF'] = [variance_inflation_factor(XRFET2.values, i) for i in range(XRFET2.shape[1])]
VIFRFET2['VIF'] = round(VIFRFET2['VIF'], 2)
VIFRFET2 = VIFRFET2.sort_values(by = "VIF", ascending = False)
VIFRFET2

## **6. Results and Inference** <font color = red>[5 marks]</font> <br>

#### **6.1 Perform Residual Analysis** <font color = red>[3 marks]</font> <br>

In [ ]:
# Perform residual analysis using plots like residuals vs predicted values, Q-Q plot and residual histogram
#residue = y_train - predicated values of y
y_train_pred_RFE = lmRFET_new.predict(X_trainRFE_new_sm)
y_train_pred_RFE

In [ ]:
resRFET = y_trainRFE - y_train_pred_RFE
sns.distplot(resRFET)
#it is centered around 0 and normal distributed

[Your inferences here:]

Residual Analysis – Inferences
The residuals are approximately normally distributed, forming a bell-shaped curve.
They are centered around zero, indicating that the model is not systematically overpredicting or underpredicting.
The spread of residuals appears fairly symmetric, suggesting that errors are balanced on both sides.
There are no major skewness or extreme irregularities, which supports the assumption of normality.
A few larger residuals are present, but these are expected in real-world data and do not significantly distort the distribution.
Conclusion

Overall, the residual distribution indicates that the linear regression assumptions are reasonably satisfied, and the model provides an unbiased and reliable fit for predicting delivery time.

#### **6.2 Perform Coefficient Analysis** <font color = red>[2 marks]</font> <br>

Perform coefficient analysis to find how changes in features affect the target.
Also, the features were scaled, so interpret the scaled and unscaled coefficients to understand the impact of feature changes on delivery time.


In [ ]:
# Compare the scaled vs unscaled features used in the final model
# Unscaled coefficients from final OLS model
coef_unscaled = lmRFET_new.params.drop('const')
print("Unscaled Coefficients:")
print(coef_unscaled)

In [ ]:
# Scale only the feature columns (drop const before scaling)
X_train_features = X_trainRFE_new.drop(columns='const')
X_train_scaled = scaler.fit_transform(X_train_features)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_features.columns)

In [ ]:
# Fit linear regression model on scaled features
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_trainRFE)

# Get scaled coefficients
coef_scaled = pd.Series(model_scaled.coef_, index=X_train_scaled.columns)
print("\nScaled Coefficients:")
print(coef_scaled)

In [ ]:
# Compare scaled vs unscaled coefficients
coef_compare = pd.DataFrame({
    'Unscaled_Coefficient': coef_unscaled,
    'Scaled_Coefficient': coef_scaled
})

coef_compare['Abs_Scaled'] = coef_compare['Scaled_Coefficient'].abs()
coef_compare = coef_compare.sort_values(by='Abs_Scaled', ascending=False)
print("\nComparison of Scaled vs Unscaled Coefficients:")
print(coef_compare)

#The scaled and unscaled coefficients are the same because feature scaling was already applied prior to model training. As a result, both models use the same scaled feature values, and reapplying scaling does not alter the coefficient magnitudes.

Additionally, we can analyse the effect of a unit change in a feature. In other words, because we have scaled the features, a unit change in the features will not translate directly to the model. Use scaled and unscaled coefficients to find how will a unit change in a feature affect the target.

In [ ]:
# Analyze the effect of a unit change in a feature, say 'total_items'

# Get coefficient of total_items
coef_total_items = coef_unscaled.get('total_items', None)

if coef_total_items is not None:
    print(f"Unscaled coefficient for total_items: {coef_total_items}")

    print("\nInterpretation:")
    print(f"A 1 unit increase in total_items changes delivery_time by {coef_total_items:.2f} units (minutes).")

else:
    print("total_items was dropped from the final model, so it does not impact the target anymore.")

In [ ]:
# Analyze the effect of a unit change in 'subtotal'

coef_subtotal = coef_unscaled['subtotal']

print(f"Unscaled coefficient for subtotal: {coef_subtotal}")

print("\nInterpretation:")
print(f"A 1 unit increase in subtotal increases delivery_time by {coef_subtotal:.2f} minutes.")

Note:
The coefficients on the original scale might differ greatly in magnitude from the scaled coefficients, but they both describe the same relationships between variables.

Interpretation is key: Focus on the direction and magnitude of the coefficients on the original scale to understand the impact of each variable on the response variable in the original units.

Include conclusions in your report document.

## Subjective Questions <font color = red>[20 marks]</font>

Answer the following questions only in the notebook. Include the visualisations/methodologies/insights/outcomes from all the above steps in your report.

#### Subjective Questions based on Assignment

##### **Question 1.** <font color = red>[2 marks]</font> <br>

Are there any categorical variables in the data? From your analysis of the categorical variables from the dataset, what could you infer about their effect on the dependent variable?

**Answer:**
>
Yes, there are categorical variables present in the dataset. The main categorical features I identified are market_id, store_primary_category, and order_protocol.

From my analysis, these variables have a noticeable impact on the dependent variable, which is delivery time.

store_primary_category has a strong effect on delivery time because different types of restaurants have different preparation times. For example, fast-food outlets tend to prepare orders faster compared to restaurants with more complex menus, which increases delivery time.
market_id also influences delivery time as it represents different geographical locations. Delivery efficiency varies across regions due to factors like traffic conditions, distance, and local demand patterns.
order_protocol affects how quickly an order is processed. Some ordering methods may be more efficient (automated systems), while others may introduce delays, which can impact delivery time.

Overall, I observed that these categorical variables capture important operational and behavioral differences in the data, and therefore, they play a significant role in explaining variations in delivery time.



---



##### **Question 2.** <font color = red>[1 marks]</font> <br>
What does `test_size = 0.2` refer to during splitting the data into training and test sets?

**Answer:**
>
test_size = 0.2 means that 20% of the total dataset is allocated to the test set, while the remaining 80% is used for training the model.

In my case, I used this split to ensure that the model is trained on the majority of the data (80%) and then evaluated on unseen data (20%) to check its performance.

The test set helps in understanding how well the model generalizes to new data and prevents overfitting, as the model has not seen this portion of the data during training.



---



##### **Question 3.** <font color = red>[1 marks]</font> <br>
Looking at the heatmap, which one has the highest correlation with the target variable?  

>


**Answer:**
>
From the heatmap, I observed that total_items has the highest correlation with the target variable (delivery_time).

This makes sense because as the number of items in an order increases, the preparation time also increases, which directly impacts the overall delivery time.

So, among the numerical features, total_items shows the strongest relationship with delivery time and is an important predictor in the model.



---



##### **Question 4.** <font color = red>[2 marks]</font> <br>
What was your approach to detect the outliers? How did you address them?

**Answer:**

>
To detect outliers, I used visualization techniques, mainly boxplots and distribution plots for the numerical variables. These helped me identify extreme values that were significantly far from the majority of the data.

In addition, I also looked at the spread and range of the data to spot unusually high values, especially in features like delivery time and total items.

To address the outliers, I did not remove them completely. Instead, I handled them by:

Understanding their impact on the model
Ensuring that they were not data entry errors
Proceeding with feature scaling, which helps reduce the influence of extreme values on the model

This approach allowed me to retain important information while minimizing the effect of outliers on model performance.

To address the outliers, the IQR (Interquartile Range) method was used. Values lying below Q1 − 1.5 × IQR and above Q3 + 1.5 × IQR were treated as outliers and removed from the dataset.




---



##### **Question 5.** <font color = red>[2 marks]</font> <br>
Based on the final model, which are the top 3 features significantly affecting the delivery time?

**Answer:**
>
Based on the final model and the coefficient analysis, the top 3 features significantly affecting delivery time are:

>total_items

This has the strongest impact, as an increase in the number of items directly increases the preparation time, leading to longer delivery times.

>subtotal

Higher order value generally indicates larger or more complex orders, which again increases preparation time and affects delivery duration.

>num_distinct_items

Orders with a higher variety of items require more effort and coordination during preparation, which contributes to increased delivery time.

Overall, these features are all related to order size and complexity, which are the primary drivers of delivery time in the model.



---



#### General Subjective Questions

##### **Question 6.** <font color = red>[3 marks]</font> <br>
Explain the linear regression algorithm in detail

**Answer:**
>
Linear Regression is a supervised machine learning algorithm that is used to model the relationship between a dependent variable (target) and one or more independent variables (features).

The goal of linear regression is to find the best-fitting straight line (or hyperplane in multiple dimensions) that predicts the target variable based on the input features.

Mathematical Representation

y=β0 +β1x1 +β2x2 +⋯+βnxn
	​


Where:y = dependent variable (delivery time in my case)
β0 = intercept
β1,β2,...,βn = coefficients
x1,x2,...,xn = independent variables




---



##### **Question 7.** <font color = red>[2 marks]</font> <br>
Explain the difference between simple linear regression and multiple linear regression

**Answer:**
>
Simple Linear Regression uses one independent variable to predict a dependent variable, whereas Multiple Linear Regression uses two or more independent variables to predict a dependent variable.
Simple linear regression is used when there is only one predictor, while multiple linear regression is used when multiple features influence the target variable.



---



##### **Question 8.** <font color = red>[2 marks]</font> <br>
What is the role of the cost function in linear regression, and how is it minimized?

**Answer:**
>
The Cost Function in linear regression measures the difference between the actual values and predicted values. It helps evaluate how well the model is performing.
The goal of linear regression is to minimize the cost function by finding the best values of coefficients.

The cost function is minimized using methods like:

Gradient Descent
Ordinary Least Squares (OLS)

Minimizing the cost function helps improve model accuracy.



---



##### **Question 9.** <font color = red>[2 marks]</font> <br>
Explain the difference between overfitting and underfitting.



**Answer:**

>
Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in high accuracy on training data but poor performance on test data.

Underfitting occurs when a model is too simple to capture the underlying pattern in the data, resulting in poor performance on both training and test data.

Overfitting → High variance
Underfitting → High bias



---



##### **Question 10.** <font color = red>[3 marks]</font> <br>
How do residual plots help in diagnosing a linear regression model?

**Answer:**
>
Role of Residual Plots in Linear Regression

Residual Analysis is used to evaluate model assumptions and performance. Residuals represent the difference between actual and predicted values.
Residual plots help in diagnosing:

Whether the linear relationship assumption is valid
Presence of outliers
Constant variance of errors (homoscedasticity)
Model performance

If residuals are randomly scattered around zero, the model is appropriate. If patterns appear, it indicates model issues.